# 08 — Real-World Analytics Pipeline
**Goal:** Build an end-to-end growth analytics pipeline using everything learned in this course.

We will:
1. Load the funnel data and extract a NumPy tensor
2. Compute step-by-step conversion rates across channels and time
3. Detect anomalies using Z-scores
4. Run a simple attribution model (weighted contribution)
5. Simulate a budget optimization scenario
6. Produce a summary report

In [ ]:
import os
os.chdir('/Volumes/SSD_Gabo/proyectos/growth-analytics')

import numpy as np
import pandas as pd

print('numpy:', np.__version__)
print('pandas:', pd.__version__)

## 1. Load data and build a (channels × days × steps) tensor

In [ ]:
df = pd.read_csv('data/raw/funnel_data.csv', parse_dates=['date'])

CHANNELS = ['organic', 'paid', 'email', 'social', 'direct']
STEPS    = ['visita_landing', 'inicio_solicitud', 'datos_personales',
            'otp', 'datos_financieros', 'carga_documentos',
            'evaluacion_crediticia', 'aprobacion', 'firma_digital',
            'activacion_tarjeta']

N_CHANNELS = len(CHANNELS)
N_STEPS    = len(STEPS)

# Build tensor: (channels, days, steps)
slices = []
for ch in CHANNELS:
    sub = (df[df['channel'] == ch]
           .sort_values('date')[STEPS]
           .to_numpy(dtype=float))
    slices.append(sub)

tensor = np.stack(slices, axis=0)  # (5, 90, 10)
print(f'Tensor shape: {tensor.shape}  →  (channels, days, steps)')
print(f'Channels: {N_CHANNELS}  |  Days: {tensor.shape[1]}  |  Steps: {N_STEPS}')

## 2. Step-by-step conversion rates

In [ ]:
# Step conversion: step[i+1] / step[i] per (channel, day)
# tensor[:, :, :-1] are the denominators, tensor[:, :, 1:] the numerators
step_cvr = tensor[:, :, 1:] / tensor[:, :, :-1]  # (5, 90, 9)

# Average step CVR per channel (over all days)
mean_step_cvr = step_cvr.mean(axis=1)  # (5, 9)

print('Mean step-by-step CVR (%) per channel:')
step_labels = [f'{STEPS[i]}→{STEPS[i+1].split("_")[0]}' for i in range(9)]
header = f'{"":10s}' + ''.join(f'{s[:12]:>14s}' for s in STEPS[1:])
print(header)
for ch, row in zip(CHANNELS, mean_step_cvr):
    vals = ''.join(f'{v*100:13.1f}%' for v in row)
    print(f'{ch:10s}{vals}')

In [ ]:
# End-to-end CVR: last step / first step
e2e_cvr = tensor[:, :, -1] / tensor[:, :, 0]  # (5, 90)

print('End-to-end CVR stats per channel:')
print(f'{"channel":10s} {"mean":>8s} {"std":>8s} {"min":>8s} {"max":>8s} {"p90":>8s}')
for i, ch in enumerate(CHANNELS):
    cvr = e2e_cvr[i]
    print(f'{ch:10s} {cvr.mean()*100:7.2f}% {cvr.std()*100:7.2f}% '
          f'{cvr.min()*100:7.2f}% {cvr.max()*100:7.2f}% '
          f'{np.percentile(cvr, 90)*100:7.2f}%')

## 3. Anomaly detection — Z-score method

In [ ]:
# Flag days where CVR is more than 2 std devs from channel mean
THRESHOLD = 2.0

# Z-score per channel (along days axis)
mean = e2e_cvr.mean(axis=1, keepdims=True)  # (5, 1)
std  = e2e_cvr.std(axis=1, keepdims=True)   # (5, 1)
z    = (e2e_cvr - mean) / std               # (5, 90)

anomaly_mask = np.abs(z) > THRESHOLD        # (5, 90)

print(f'Anomalies (|z| > {THRESHOLD}) per channel:')
for i, ch in enumerate(CHANNELS):
    days_flagged = anomaly_mask[i].sum()
    print(f'  {ch:8s}: {days_flagged} day(s) flagged')

print(f'\nTotal anomaly-days: {anomaly_mask.sum()}')

In [ ]:
# Which channel/day has the worst drop?
flat_z = z.flatten()
worst_idx = np.unravel_index(np.argmin(flat_z), z.shape)
ch_idx, day_idx = worst_idx

print(f'Worst CVR drop: channel={CHANNELS[ch_idx]}, day={day_idx}')
print(f'  Z-score: {z[ch_idx, day_idx]:.2f}')
print(f'  CVR: {e2e_cvr[ch_idx, day_idx]*100:.3f}%  '
      f'(mean: {e2e_cvr[ch_idx].mean()*100:.3f}%)')

## 4. Attribution model — weighted channel contribution

In [ ]:
# Total sessions and activations per channel
total_sessions    = tensor[:, :, 0].sum(axis=1)   # (5,)
total_activations = tensor[:, :, -1].sum(axis=1)  # (5,)

# Session-share weights
session_share = total_sessions / total_sessions.sum()  # (5,)

# Contribution = activations * session_share (volume-weighted attribution)
channel_cvr = total_activations / total_sessions
weighted_contribution = channel_cvr * session_share
blended_cvr = weighted_contribution.sum()

print('Channel attribution:')
print(f'{"channel":10s} {"sessions":>12s} {"share":>8s} {"CVR":>8s} {"contribution":>14s}')
for i, ch in enumerate(CHANNELS):
    print(f'{ch:10s} {total_sessions[i]:>12,.0f} '
          f'{session_share[i]*100:7.1f}% '
          f'{channel_cvr[i]*100:7.2f}% '
          f'{weighted_contribution[i]*100:13.3f}%')
print(f'{"BLENDED":10s} {total_sessions.sum():>12,.0f}         '
      f'        {blended_cvr*100:13.3f}%')

## 5. Budget simulation — marginal returns

In [ ]:
# Assume a cost-per-session for each channel (CPS)
cps = np.array([0.5, 2.0, 1.0, 0.8, 0.3])   # USD per session

# Current monthly budget allocation (proportional to sessions)
budget_total = 50_000  # USD
budget = session_share * budget_total  # distribute by session share

# Projected sessions from budget / CPS
projected_sessions = budget / cps

# Projected activations
projected_activations = projected_sessions * channel_cvr

# What if we shift 10% budget from paid to organic?
shift = 0.10 * budget_total
budget_new = budget.copy()
budget_new[0] += shift   # add to organic
budget_new[1] -= shift   # take from paid

projected_sessions_new    = budget_new / cps
projected_activations_new = projected_sessions_new * channel_cvr

delta_activations = projected_activations_new.sum() - projected_activations.sum()

print(f'Current projected activations:  {projected_activations.sum():>8,.0f}')
print(f'After shift projected:           {projected_activations_new.sum():>8,.0f}')
print(f'Delta:                           {delta_activations:>+8,.0f}')
print(f'% change:                        {delta_activations/projected_activations.sum()*100:>+7.2f}%')

## 6. Summary report

In [ ]:
print('=' * 60)
print('GROWTH ANALYTICS — Q1 FUNNEL REPORT')
print('=' * 60)
print(f'Period:               {tensor.shape[1]} days')
print(f'Total sessions:       {total_sessions.sum():>12,.0f}')
print(f'Total activations:    {total_activations.sum():>12,.0f}')
print(f'Blended E2E CVR:      {blended_cvr*100:>11.3f}%')
print()
print('Top channel by CVR:   ', CHANNELS[np.argmax(channel_cvr)])
print('Top channel by volume:', CHANNELS[np.argmax(total_sessions)])
print()
print(f'Anomalous days (|z|>{THRESHOLD}): {anomaly_mask.sum()} across all channels')
print('=' * 60)

print('\nChannel snapshot:')
print(f'{"channel":10s} {"sessions":>10s} {"activations":>12s} {"CVR":>8s}')
for i, ch in enumerate(CHANNELS):
    print(f'{ch:10s} {total_sessions[i]:>10,.0f} '
          f'{total_activations[i]:>12,.0f} '
          f'{channel_cvr[i]*100:>7.2f}%')

## What we used in this notebook
| Topic | Applied as |
|---|---|
| Array creation | `np.stack` to build the 3D tensor |
| Indexing | `tensor[:, :, -1]` to access last funnel step |
| Broadcasting | `(e2e_cvr - mean) / std` with `keepdims=True` |
| Aggregations | `.sum(axis=1)`, `.mean()`, `np.percentile` |
| Boolean indexing | `anomaly_mask = np.abs(z) > threshold` |
| `np.unravel_index` | Map flat argmin back to (channel, day) |
| Linear algebra | Weighted dot product for blended CVR |
| Simulation | Vectorized budget reallocation scenario |

---
**Course complete.** You now know enough NumPy to work comfortably with pandas internals, feed data into scikit-learn, and build fast analytics pipelines without writing Python loops.